# DADOS tutorial
Tutorial to get a aquanted with the DADOS data reduction pipeline

In [ ]:
# Step 0
########
# Load some required libraries
import numpy as np
import matplotlib.pyplot as plt
from astropy.io.fits import getdata, getheader
from dared.utils import Formatter
fmt = Formatter()

# Step 1
########
# Load the data files

flat_frame        = "flats.fit"
calibration_frame = "wavelength_calib.fit"

# Import, initialize, and calibrate the spectrograph observation object
from dared.dados_red import DADOSobservation, AlpyArNe

# initialize the observation object
obs = DADOSobservation(
    flat=getdata(flat_frame),
    calib=getdata(calibration_frame),
    calib_lamp=AlpyArNe
)
obs.debug_mode = True # enables the diagnostic plots to see whether the calibration worked out correctly

fmt.info("Current DADOS observation status:", obs.status)

# run the analysis to find the spectrographs orders
obs.findorder()
fmt.info("Current DADOS observation status:", obs.status) # now the oders should have been found ideally

In [ ]:
# read out the entire spectrum slit order (e.g. the middle slit only) and integrate along the entire slit width
# extract the fluxes in both the wavelength calibration image and the science image consistently

science_image_file = "testspec/spectra/spectrum-0001_science_300s.fit"

#science_image_fluxes = obs.extract_flux(
#    getdata(science_image_file),
#    aperture_offsets=0,  # keep this parameter consistent between the science image and the wavelength calibration image
#    aperture_height=None # keep this parameter consistent between the science image and the wavelength calibration image
#)

#calibration_image_fluxes = obs.extract_flux(
#    getdata(calibration_frame),
#    aperture_offsets=0,  # keep this parameter consistent between the science image and the wavelength calibration image
#    aperture_height=None # keep this parameter consistent between the science image and the wavelength calibration image
#)

spectra = obs.get_spectrum(
    getdata(science_image_file),
    readout_kwargs=dict(
        aperture_offsets=0., # uses the vertical center of the spectral orders
        aperture_height=None, # defaults to the maximal order width
        order=1
    )
)

In [ ]:
# print the output
fig, ax = plt.subplots()
ax.step(*spectra[0].T, color="k", lw=0.5)
ax.set_xlabel(r"Wavelength $\lambda$ [Å]")
ax.set_ylabel(r"Normalized Flux $F_\lambda$ a.u.")
plt.show()